# 1.2 Raw Data Preprocessing

Clean and validate the raw data, then save the preprocessed train and test versions.

**Steps:**
1. Load raw train and test splits
2. Apply the same preprocessing pipeline to both splits
3. Detect and handle outliers (train)
4. Create house_age feature (train)
5. Validate data integrity (both splits)
6. Save preprocessed train and test data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

## 1.2.1 Load Raw Data (Train + Test)

In [ ]:
RAW_TRAIN_PATH = '../data/raw/data_raw_train.csv'
RAW_TEST_PATH = '../data/raw/data_raw_test.csv'
PROCESSED_DIR = '../data/processed'

os.makedirs(PROCESSED_DIR, exist_ok=True)

df_train = pd.read_csv(RAW_TRAIN_PATH)
df_test = pd.read_csv(RAW_TEST_PATH)
print(f'Loaded {df_train.shape[0]} train rows and {df_train.shape[1]} columns')
print(f'Loaded {df_test.shape[0]} test rows and {df_test.shape[1]} columns')
df_train.head()

## 1.2.2 Shared Preprocessing Pipeline

In [ ]:
def detect_outliers_iqr(series, factor=1.5):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    return (series < lower) | (series > upper)


def cap_outliers(series, factor=1.5):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    return series.clip(lower, upper)


outlier_cols = ['price', 'sqft']
CURRENT_YEAR = 2026


def preprocess(df):
    df = df.copy()

    initial_rows = len(df)
    df = df.dropna(subset=['price'])
    print(f'Dropped {initial_rows - len(df)} rows with missing price.')

    numerical_cols = df.select_dtypes(include=[np.number]).columns
    for col in numerical_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].median())
            print(f'Filled {col} missing values with median: {df[col].median():.2f}')

    categorical_cols = df.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        if df[col].isnull().sum() > 0:
            df[col] = df[col].fillna(df[col].mode()[0])
            print(f'Filled {col} missing values with mode: {df[col].mode()[0]}')

    df['price'] = df['price'].astype(float)
    df['sqft'] = df['sqft'].astype(int)
    df['bedrooms'] = df['bedrooms'].astype(int)
    df['bathrooms'] = df['bathrooms'].astype(float)
    df['year_built'] = df['year_built'].astype(int)

    for col in outlier_cols:
        before = df[col].describe()
        df[col] = cap_outliers(df[col])
        after = df[col].describe()
        print(f'{col}: capped outliers. Range changed from [{before["min"]:.0f}, {before["max"]:.0f}] to [{after["min"]:.0f}, {after["max"]:.0f}]')

    df['house_age'] = CURRENT_YEAR - df['year_built']
    return df


df_train = preprocess(df_train)
df_test = preprocess(df_test)

print(f'\nRemaining missing values (train): {df_train.isnull().sum().sum()}')
print(f'Remaining missing values (test):  {df_test.isnull().sum().sum()}')

## 1.2.3 Data Types (Train)

In [ ]:
print('Updated dtypes (train):')
print(df_train.dtypes)

## 1.2.4 Detect and Handle Outliers (Train)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, col in enumerate(outlier_cols):
    outlier_mask = detect_outliers_iqr(df_train[col])
    n_outliers = outlier_mask.sum()
    print(f'{col}: {n_outliers} outliers detected ({n_outliers/len(df_train)*100:.1f}%)')
    sns.boxplot(x=df_train[col], ax=axes[i])
    axes[i].set_title(f'{col} Distribution (Boxplot, train)')

plt.tight_layout()
plt.show()

## 1.2.5 Feature: Age of House (Train)

In [ ]:
print('house_age statistics (train):')
print(df_train['house_age'].describe())

## 1.2.6 Data Validation (Train + Test)

In [ ]:
def validate(df):
    assert df['price'].min() > 0, 'Price has non-positive values!'
    assert df['sqft'].min() > 0, 'Sqft has non-positive values!'
    assert df['bedrooms'].min() >= 0, 'Bedrooms has negative values!'
    assert df['bathrooms'].min() >= 0, 'Bathrooms has negative values!'
    assert df['year_built'].min() >= 1900, 'year_built has suspicious values!'
    assert df['year_built'].max() <= CURRENT_YEAR, 'year_built has future dates!'
    assert df['location'].isin(['Suburb', 'Downtown', 'Rural', 'Waterfront', 'Urban', 'Mountain']).all(), 'Unexpected location values!'
    assert df['condition'].isin(['Poor', 'Fair', 'Good', 'Excellent']).all(), 'Unexpected condition values!'


validate(df_train)
validate(df_test)
print('All data validation checks passed (train and test).')

## 1.2.7 Save Preprocessed Data

In [ ]:
train_output_path = os.path.join(PROCESSED_DIR, 'house_data_preprocessed_train.csv')
test_output_path = os.path.join(PROCESSED_DIR, 'house_data_preprocessed_test.csv')

df_train.to_csv(train_output_path, index=False)
df_test.to_csv(test_output_path, index=False)

print(f'Saved preprocessed train data to {train_output_path}')
print(f'Train shape: {df_train.shape}')
print(f'Saved preprocessed test data to {test_output_path}')
print(f'Test shape: {df_test.shape}')
df_train.head()